## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

from neuro_fuzzy_toolbox import h_ANFIS, Hybrid_learning_algorithm, SONFIS, EarlyStopping, get_measures, Gaussian_MF, Optimizer_training

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [4]:
from ucimlrepo import fetch_ucirepo

# Heart Disease

## Data

In [5]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [6]:
X = X.fillna(value=X.mean())

In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [8]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [9]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [10]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [11]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [12]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [13]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 16, shuffle = True)
x_train = loader.dataset.tensors[0]
y_train = loader.dataset.tensors[1]
x_train.shape, y_train.shape

(torch.Size([212, 13]), torch.Size([212]))

In [14]:
x_train

tensor([[-0.3333,  1.0000,  1.0000,  ...,  0.0000,  1.0000,  1.0000],
        [-0.1250,  1.0000,  0.3333,  ...,  0.0000, -0.3333,  1.0000],
        [ 0.4167,  1.0000,  1.0000,  ...,  0.0000, -0.3333,  1.0000],
        ...,
        [-0.4583,  1.0000, -1.0000,  ..., -1.0000,  0.3333, -1.0000],
        [ 0.9583, -1.0000,  0.3333,  ...,  0.0000, -1.0000, -1.0000],
        [ 0.0417,  1.0000, -0.3333,  ..., -1.0000, -1.0000,  1.0000]],
       dtype=torch.float64)

In [15]:
y_train

tensor([3, 1, 2, 3, 4, 1, 1, 0, 1, 0, 1, 3, 0, 3, 0, 0, 1, 0, 0, 0, 1, 2, 0, 1,
        0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 3, 4, 0, 2, 1, 3, 4, 2, 4, 0, 3, 0,
        0, 3, 1, 0, 0, 0, 0, 0, 2, 1, 0, 1, 0, 0, 0, 0, 0, 2, 1, 0, 1, 1, 0, 3,
        0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 1, 0, 3, 0, 2, 0, 0, 2, 0, 4, 0, 0, 3,
        0, 0, 2, 4, 0, 3, 0, 0, 1, 2, 1, 3, 4, 1, 0, 1, 3, 0, 0, 0, 1, 3, 2, 4,
        1, 2, 0, 0, 0, 2, 0, 0, 3, 3, 0, 0, 2, 0, 1, 2, 0, 2, 0, 3, 4, 0, 0, 2,
        2, 2, 0, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 1, 0, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 2, 2, 1, 2, 0, 1, 3, 0, 0, 0, 0, 1, 3, 3, 1, 0, 1, 2,
        0, 0, 1, 0, 0, 3, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

## Model & Training

### ANFIS

In [16]:
model = h_ANFIS(
    input_size = 13,
    num_mfs = 1,
    outputs = 5,
    membership_function=Gaussian_MF,
    rule_reduced = True,
    output_type='multiclass'
)

model.init_premises(x_train)

### Hybrid Learning Algorithm

In [ ]:
loss_fn = nn.functional.cross_entropy
epochs = 500
optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.Adam
#params = {'lr': 0.0001}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=50
)

In [18]:
#trainer = Hybrid_learning_algorithm(
#    epochs=epochs,
#    loss_function=loss_fn,
#    optimizer=optimizer,
#    optimizer_params=params,
#    early_stopping=early_stopping
#)

trainer = Optimizer_training(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [19]:
#Ngrow = 30
#dGrow = 0.8
#Nsplit = 30
#eSplit = 0.7
#Nvanish = 8
#lVanish = 3

#Ngrow = 8
#dGrow = 3.0
#Nsplit = 6
#eSplit = 2.6
#Nvanish = 3
#lVanish = 3

Ngrow = 20
dGrow = 1.5
Nsplit = 15
eSplit = 1.1
Nvanish = 8
lVanish = 4

max_iterations = 100

anfis_trainer = trainer

validation = 0.3
sonfis_early_stopping = EarlyStopping(patience=15)
last_training_iteration = True

In [20]:
sonfis = SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [21]:
%time sonfis(model, loader, verbose=True)

Iteration:   0/100 - loss: 3.669631 - validation loss: 3.664441
 -> ANFIS rules: 1

Iteration:   1/100 - loss: 1.354512 - validation loss: 1.450067
 -> ANFIS rules: 2

Iteration:   2/100 - loss: 1.281093 - validation loss: 1.379729
 -> ANFIS rules: 4

Iteration:   3/100 - loss: 1.225854 - validation loss: 1.338427
 -> ANFIS rules: 7

Iteration:   4/100 - loss: 1.225854 - validation loss: 1.338427
 -> ANFIS rules: 8

Iteration:   5/100 - loss: 1.222638 - validation loss: 1.311611
 -> ANFIS rules: 8

Iteration:   6/100 - loss: 1.222638 - validation loss: 1.311611
 -> ANFIS rules: 9

Iteration:   7/100 - loss: 1.222638 - validation loss: 1.311611
 -> ANFIS rules: 10

Iteration:   8/100 - loss: 1.222638 - validation loss: 1.311611
 -> ANFIS rules: 11

Iteration:   9/100 - loss: 1.222638 - validation loss: 1.311611
 -> ANFIS rules: 11

Iteration:  10/100 - loss: 1.222638 - validation loss: 1.311611
 -> ANFIS rules: 12

Iteration:  11/100 - loss: 1.222638 - validation loss: 1.311611
 -> ANFI

In [22]:
test_measures = get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.45054945054945056
Precision: 0.40458747601604744
Recall: 0.45054945054945056
F1: 0.4252616653295771
Confusion Matrix: [[36  9  2  0  2]
 [ 8  2  3  3  1]
 [ 6  0  0  4  1]
 [ 4  3  0  2  1]
 [ 2  0  1  0  1]]


In [23]:
train_measures = get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.6745283018867925
Precision: 0.6515219357355654
Recall: 0.6745283018867925
F1: 0.6567622698768507
Confusion Matrix: [[103  10   2   0   0]
 [ 20  13   2   2   1]
 [  6   3  10   4   2]
 [  3   5   1  15   1]
 [  1   1   1   4   2]]
